## Introduction to Econometrics
Prof. Roberto Renò - ESSEC Business School 

Based on:
**[gabors-data-analysis.com ](https://gabors-data-analysis.com/)**

 License: Free to share, modify and use for educational purposes. 
 Not to be used for commercial purposes.

## Class 7

## Multiple Regression

In [1]:
import os
import sys
import warnings

import numpy as np
import pandas as pd
from mizani.formatters import percent_format
from plotnine import *
from datetime import datetime
from scipy.stats import norm
import statsmodels.api as sm
import statsmodels.formula.api as smf
from mizani import transforms
from stargazer.stargazer import Stargazer
from IPython.core.display import HTML

warnings.filterwarnings("ignore")

# Current script folder
current_path = os.getcwd()
dirname = current_path.split("da_case_studies")[0]

# location folders (double check you have this on your pc)
data_in = dirname + "/Data/"
data_out = dirname + "/Data/"
output = dirname + "/output/"
func = dirname + "/Settings/"
sys.path.append(func)

# Import the prewritten helper functions
from py_helper_functions import *

In [2]:
# Import the prewritten helper functions 
from py_helper_functions import *

In [3]:
cps = pd.read_csv(data_in + "morg-2014-emp.csv")

### grade92

Item 18h. Highest grade completed. “What is the highest level of school ... has completed or highest degree received?” In 1992 the BLS switched from years of schooling measure to a credential oriented measure. Rumor has it that a labor economist who estimated wage equations for 1991 and 1992 without noticing the difference in the CPS education measure was surprised only by the change in the constant term. Imputed highest grade completed is available 1998 on in ihigrdc.
Less than 1st grade 31
10th 11th
grade 32 33 34 35 36 37 12th grade NO DIPLOMA 38 High school graduate, diploma or GED 39 Some college but no degree 40 Associate degree -- occupational/vocational 41 Associate degree -- academic program 42 Bachelor's degree (e.g. BA,AB,BS) 43 Master's degree (e.g. MA,MS,MEng,Med,MSW,MBA) 44 Professional school deg. (e.g. MD,DDS,DVM,LLB,JD) 45 Doctorate degree (e.g. PhD, EdD) 46

In [4]:
cps = cps.query("uhours>=20 & earnwke>0 & age>=24 & age<=64 & grade92>=44")

In [5]:
# CREATE VARIABLES
cps["female"] = (cps.sex == 2).astype(int)
cps["w"] = cps["earnwke"] / cps["uhours"]
cps["lnw"] = np.log(cps["w"])

## Write out to csv
cps.to_csv(data_out + "earnings_multireg.csv")

In [6]:
#####################
#DISTRIBUTION OF EARNINGS
#######################
cps.loc[:,["earnwke","uhours","w"]].describe()

,earnwke,uhours,w
count,18241.00000,18241.000000,18241.000000
mean,1481.78936,42.970780,34.525791
std,747.92426,9.139368,16.654215
min,0.01000,20.000000,0.000200
25%,923.00000,40.000000,21.634500
50%,1346.00000,40.000000,31.250000
75%,1923.07000,47.000000,45.673000
max,2884.61000,99.000000,144.230500


In [7]:
cps.loc[cps.w>=1,["earnwke","uhours","w"]].describe()

,earnwke,uhours,w
count,18220.000000,18220.000000,18220.000000
mean,1483.491212,42.970088,34.565432
std,746.672256,9.135281,16.622801
min,38.000000,20.000000,1.025556
25%,923.000000,40.000000,21.634500
50%,1346.000000,40.000000,31.250000
75%,1923.070000,47.000000,45.673000
max,2884.610000,99.000000,144.230500


### Table 10.1 Gender differences in earnings – log earnings and gender

In [8]:
# use robust std
reg = smf.ols(formula="lnw~female", data=cps).fit(cov_type="HC1")
reg2 = smf.ols(formula="lnw~female+age", data=cps).fit(cov_type="HC1")
reg3 = smf.ols(formula="age~female", data=cps).fit(cov_type="HC1")

In [9]:
stargazer = Stargazer([reg, reg2, reg3])
stargazer.custom_columns(["ln wage", "ln wage", "age"], [1, 1, 1])
stargazer.covariate_order(
    ["female", "age", "Intercept"]
)
stargazer.rename_covariates({"Intercept": "Constant"})
stargazer

In [14]:
reg.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                    lnw   R-squared:                       0.028
Model:                            OLS   Adj. R-squared:                  0.028
Method:                 Least Squares   F-statistic:                     531.5
Date:                Wed, 26 Mar 2025   Prob (F-statistic):          5.96e-116
Time:                        10:40:28   Log-Likelihood:                -15672.
No. Observations:               18241   AIC:                         3.135e+04
Df Residuals:                   18239   BIC:                         3.136e+04
Df Model:                           1                                         
Covariance Type:                  HC1                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      3.5145      0.006    573.388      0.000       3.502       3.526
female        -0.1953      0.008    -23.055      0.000      -0.212      -0.179
==============================================================================
Omnibus:                    15252.403   Durbin-Watson:                   1.881
Prob(Omnibus):                  0.000   Jarque-Bera (JB):          1835979.169
Skew:                          -3.402   Prob(JB):                         0.00
Kurtosis:                      51.676   Cond. No.                         2.70
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity robust (HC1)
"""

In [13]:
reg2.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                    lnw   R-squared:                       0.046
Model:                            OLS   Adj. R-squared:                  0.045
Method:                 Least Squares   F-statistic:                     447.3
Date:                Wed, 26 Mar 2025   Prob (F-statistic):          2.20e-190
Time:                        10:33:10   Log-Likelihood:                -15509.
No. Observations:               18241   AIC:                         3.102e+04
Df Residuals:                   18238   BIC:                         3.105e+04
Df Model:                           2                                         
Covariance Type:                  HC1                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      3.1978      0.018    176.797      0.000       3.162       3.233
female        -0.1847      0.008    -21.937      0.000      -0.201      -0.168
age            0.0071      0.000     18.308      0.000       0.006       0.008
==============================================================================
Omnibus:                    15618.290   Durbin-Watson:                   1.875
Prob(Omnibus):                  0.000   Jarque-Bera (JB):          2007068.681
Skew:                          -3.521   Prob(JB):                         0.00
Kurtosis:                      53.903   Cond. No.                         201.
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity robust (HC1)
"""

In [ ]:
# AGE distribution (histogram)
ggplot(cps, aes(x="age", y="2*stat(count)/sum(stat(count))")) + geom_histogram(
    binwidth=4,
    color="white",
    fill=color[0],
    size=0.25,
    alpha=0.8,
    show_legend=False,
    na_rm=True,
) + labs(x="Age (years)", y="Percent") + facet_wrap('~female',labeller={'0':"Male",'1':"Female"}
) + labs(
    x="Age (years)", y="Percent"
) + scale_x_continuous(
    limits=(24, 64), breaks=seq(25, 65, by=5),
) + scale_y_continuous(
    limits=(0, 0.16), breaks=seq(0, 0.16, by=0.02), labels=percent_format()
) + theme_bw()

In [ ]:
# age distribution (density)
ggplot(cps, aes(x="age", y="stat(density)", color="factor(female)")) + geom_density(
    adjust=1.5, show_legend=False, na_rm=True, size=0.7
) + labs(x="Age (years)", y="Density", color=""
) + scale_color_manual(
    name="", values=(color[1], color[0]), labels=("Male", "Female")
) + scale_x_continuous(
    expand=(0.01, 0.01), limits=(24, 64), breaks=seq(25, 65, by=5)
) + scale_y_continuous(
    expand=(0.0, 0.0), limits=(0, 0.04), breaks=seq(0, 0.04, by=0.01)
) + geom_text(
    aes(x=55, y=0.028, label="'Male'"), color=color[1], size=12
) + geom_text(
    aes(x=55, y=0.020, label="'Female'"), color=color[0], size=12
) + theme_bw()

### Table 10.2 Gender differences in earnings – log earnings and age, various functional forms

In [ ]:
cps["agesq"] = np.power(cps["age"], 2)/40**2
cps["agecu"] = np.power(cps["age"], 3)/40**3
cps["agequ"] = np.power(cps["age"], 4)/40**4

In [ ]:
reg4 = smf.ols(formula="lnw~female", data=cps).fit(cov_type="HC1")
reg5 = smf.ols(formula="lnw~female+age", data=cps).fit(cov_type="HC1")
reg6 = smf.ols(formula="lnw~female+age+agesq", data=cps).fit(cov_type="HC1")
reg7 = smf.ols(formula="lnw~female+age+agesq+agecu+agequ", data=cps).fit(cov_type="HC1")

In [ ]:
stargazer = Stargazer([reg4,reg5,reg6,reg7])
stargazer.covariate_order(
    ["female", "age", "agesq","agecu","agequ","Intercept"]
)
stargazer.rename_covariates({"Intercept": "Constant"})
stargazer

### Table 10.3 Gender differences in earnings – log earnings, gender and education

In [ ]:
# create dummy variables
cps["ed_MA"] = (cps["grade92"] == 44).astype(int)
cps["ed_Profess"] = (cps["grade92"] == 45).astype(int)
cps["ed_Phd"] = (cps["grade92"] == 46).astype(int)

In [ ]:
reg8 = smf.ols(formula="lnw~female", data=cps).fit(cov_type="HC1")
reg9 = smf.ols(formula="lnw~female + ed_Profess + ed_Phd", data=cps).fit(cov_type="HC1")
reg10 = smf.ols(formula="lnw~female + ed_Profess + ed_MA", data=cps).fit(cov_type="HC1")

In [ ]:
stargazer = Stargazer([reg8,reg9,reg10])
stargazer.covariate_order(
    ["female", "ed_Profess", "ed_Phd","ed_MA","Intercept"]
)
stargazer.rename_covariates({"Intercept": "Constant"})
stargazer

### Table 10.4 Gender differences in earnings – log earnings, gender, age, and their interaction

In [ ]:
reg11 = smf.ols(formula="lnw~age", data=cps.query("female==1")).fit(cov_type="HC1")
reg12 = smf.ols(formula="lnw~age", data=cps.query("female==0")).fit(cov_type="HC1")
reg13 = smf.ols(formula="lnw~female+age+age*female", data=cps).fit(cov_type="HC1")

In [ ]:
stargazer = Stargazer([reg11, reg12, reg13])
stargazer.covariate_order(["female", "age", "age:female", "Intercept"])
stargazer.rename_covariates({"Intercept": "Constant", "age:female": "female x age"})
stargazer.custom_columns(["Women", "Men", "All"], [1, 1, 1])
stargazer

### Figure 10.2 Earning differences by gender as function of age
FOR PREDICTIONAL FUNCTIONAL FORMS & INTERACTIONS WITH GENDER

In [ ]:
reg14 = smf.ols(formula="lnw~age+agesq+agecu+agequ", data=cps.query("female==1")).fit(
    cov_type="HC1"
)
reg15 = smf.ols(formula="lnw~age+agesq+agecu+agequ", data=cps.query("female==0")).fit(
    cov_type="HC1"
)
reg16 = smf.ols(
    formula="lnw ~ age + agesq + agecu + agequ + female + female*age + female*agesq + female*agecu + female*agequ",
    data=cps,
).fit(cov_type="HC1")

In [ ]:
Stargazer([reg14,reg15,reg16])

In [ ]:
# PREDICTION AND GRAPH LINEAR
data_m=cps.query("female==0")

pred=reg13.predict(data_m)

pred = reg13.get_prediction(data_m).summary_frame()[["mean", "mean_se"]]
pred.columns = ["fit", "fit_se"]

data_m = data_m.reset_index(drop=True).join(pred)

data_m["CIup"]=data_m["fit"]+2*data_m["fit_se"]
data_m["CIlo"]=data_m["fit"]-2*data_m["fit_se"]

In [ ]:
data_f=cps.query("female==1")

pred=reg13.predict(data_f)

pred = reg13.get_prediction(data_f).summary_frame()[["mean", "mean_se"]]
pred.columns = ["fit", "fit_se"]

data_f = data_f.reset_index(drop=True).join(pred)

data_f["CIup"]=data_f["fit"]+2*data_f["fit_se"]
data_f["CIlo"]=data_f["fit"]-2*data_f["fit_se"]

In [ ]:
ggplot(data_m, aes(x="age", y="fit")) + geom_line(
    colour=color[0]
) + geom_line(
    data_m, aes(x="age", y="CIup"), colour=color[0], linetype="dashed"
) + geom_line(
    data_m, aes(x="age", y="CIlo"), colour=color[0], linetype="dashed"
) + geom_line(
    data_f, aes(x="age", y="fit"), colour=color[1]
) + geom_line(
    data_f, aes(x="age", y="CIup"), colour=color[1], linetype="dashed"
) + geom_line(
    data_f, aes(x="age", y="CIlo"), colour=color[1], linetype="dashed"
) + labs(
    x="Age (years)", y="ln(earnings per hour, US dollars)"
) + scale_x_continuous(
    expand=(0.01, 0.01), limits=(24, 65), breaks=seq(25, 66, by=5)
) + scale_y_continuous(
    expand=(0.01, 0.01), limits=(2.8, 3.8), breaks=seq(2.8, 3.9, by=0.1)
) + theme_bw()

In [ ]:
# PREDICTION AND GRAPH POLYNOMIAL
#male
data_m=cps.query("female==0")

pred=reg16.predict(data_m)

pred = reg16.get_prediction(data_m).summary_frame()[["mean", "mean_se"]]
pred.columns = ["fit", "fit_se"]

data_m = data_m.reset_index(drop=True).join(pred)

data_m["CIup"]=data_m["fit"]+2*data_m["fit_se"]
data_m["CIlo"]=data_m["fit"]-2*data_m["fit_se"]

#female
data_f=cps.query("female==1")

pred=reg16.predict(data_f)

pred = reg16.get_prediction(data_f).summary_frame()[["mean", "mean_se"]]
pred.columns = ["fit", "fit_se"]

data_f = data_f.reset_index(drop=True).join(pred)

data_f["CIup"]=data_f["fit"]+2*data_f["fit_se"]
data_f["CIlo"]=data_f["fit"]-2*data_f["fit_se"]

In [ ]:
ggplot(data_m, aes(x="age", y="fit")) + geom_line(
    colour=color[0]
) + geom_line(
    data_m, aes(x="age", y="CIup"), colour=color[0], linetype="dashed"
) + geom_line(
    data_m, aes(x="age", y="CIlo"), colour=color[0], linetype="dashed"
) + geom_line(
    data_f, aes(x="age", y="fit"), colour=color[1]
) + geom_line(
    data_f, aes(x="age", y="CIup"), colour=color[1], linetype="dashed"
) + geom_line(
    data_f, aes(x="age", y="CIlo"), colour=color[1], linetype="dashed"
) + labs(
    x="Age (years)", y="ln(earnings per hour, US dollars)"
) + scale_x_continuous(
    expand=(0.01, 0.01), limits=(24, 65), breaks=seq(25, 66, by=5)
) + scale_y_continuous(
    expand=(0.01, 0.01), limits=(2.8, 3.8), breaks=seq(2.8, 3.9, by=0.1)
) + theme_bw()

## Part 2
TOWARDS CAUSAL ANALYIS - IS IT DISCRIMINATION?

In [ ]:
# FILTER DATA -  SELECTION of the sample we need
cps = cps.query("age>=40 & age<=60")

In [ ]:
cps["white"] = (cps["race"] == 1).astype(int)
cps["afram"] = (cps["race"] == 2).astype(int)
cps["asian"] = (cps["race"] == 4).astype(int)
cps["hisp"] = (cps["ethnic"].notna()).astype(int)
cps["othernonw"] = (
    (cps["white"] == 0) & (cps["afram"] == 0) & (cps["asian"] == 0) & (cps["hisp"] == 0)
).astype(int)
cps["nonUSborn"] = (
    (cps["prcitshp"] == "Foreign Born, US Cit By Naturalization")
    | (cps["prcitshp"] == "Foreign Born, Not a US Citizen")
).astype(int)

In [ ]:
# Potentially endogeneous demographics
cps['married']=((cps['marital']==1)|(cps['marital']==2)).astype(int)
cps['divorced']=((cps['marital']==3)&(cps['marital']==5)).astype(int)
cps['wirowed']=(cps['marital']==4).astype(int)
cps['nevermar']=(cps['marital']==7).astype(int)

cps['child0']=(cps['chldpres']==0).astype(int)
cps['child1']=(cps['chldpres']==1).astype(int)
cps['child2']=(cps['chldpres']==2).astype(int)
cps['child3']=(cps['chldpres']==3).astype(int)
cps['child4pl']=(cps['chldpres']>=4).astype(int)

# Work-related variables
cps['fedgov']=(cps['class']=="Government - Federal").astype(int)
cps['stagov']=(cps['class']=="Government - State").astype(int)
cps['locgov']=(cps['class']=="Government - Local").astype(int)
cps['nonprof']=(cps['class']=="Private, Nonprofit").astype(int)
cps['ind2dig']=((pd.Categorical(cps["ind02"]).codes+1)/100).astype(int)
cps['occ2dig']=(cps["occ2012"]/100).astype(int)
cps['union']=((cps['unionmme']=="Yes")|(cps['unioncov']=="Yes")).astype(int)


In [ ]:
cps['uhourssq']=np.power(cps['uhours'],2)
cps['uhourscu']=np.power(cps['uhours'],3)
cps['uhoursqu']=np.power(cps['uhours'],4)

### Table 10.5 Gender differences in earnings – regression with many covariates on a narrower sample

In [ ]:
# Extended regressions
reg1 = smf.ols(formula="lnw ~ female", data=cps).fit(cov_type="HC1")
reg2 = smf.ols(formula="lnw ~ female + age + ed_Profess + ed_Phd", data=cps).fit(
    cov_type="HC1"
)
reg3 = smf.ols(
    formula="lnw ~ female + age + afram + hisp + asian + othernonw + nonUSborn + ed_Profess + ed_Phd + married + divorced+ wirowed + child1 + child2 + child3 +child4pl + C(stfips) + uhours + fedgov + stagov + locgov + nonprof + union + C(ind2dig) + C(occ2dig)",
    data=cps,
).fit(cov_type="HC1")
reg4 = smf.ols(
    formula="lnw ~ female + age + afram + hisp + asian + othernonw + nonUSborn + ed_Profess + ed_Phd + married + divorced+ wirowed + child1 + child2 + child3 +child4pl + C(stfips) + uhours + fedgov + stagov + locgov + nonprof + union + C(ind2dig) + C(occ2dig) + agesq + agecu + agequ + uhoursqu + uhourscu + uhourssq",
    data=cps,
).fit(cov_type="HC1")

In [ ]:
stargazer = Stargazer([reg1, reg2, reg3, reg4])
stargazer.covariate_order(["female"])
stargazer.add_line("Age and education", ["", "Yes", "Yes", "Yes"])
stargazer.add_line("Family circumstances", ["", "", "Yes", "Yes"])
stargazer.add_line("Demographic background", ["", "", "Yes", "Yes"])
stargazer.add_line("Job characteristics", ["", "", "Yes", "Yes"])
stargazer.add_line("Age in polynomial", ["", "", "", "Yes"])
stargazer.add_line("Hours in polynomial", ["", "", "", "Yes"])
stargazer